# Studio Framework Tests

Experiments and test runs against the local CCoP 2.0 evaluation stack.

The first cell is self-contained: it sets `device=mps`, brings the stack up via
`start_local.sh` (if down), and runs the deep smoke test. Execute it once before
running any subsequent test cells.

**Kernel:** `studio-ssdlc (poetry)`.


In [ ]:
# =====================================================================
# Prerequisites: device=mps + stack startup + health check
# =====================================================================
import os
import subprocess
import sys
from pathlib import Path

import torch
from dotenv import load_dotenv

# -- 0. Paths -----------------------------------------------------------
REPO_ROOT = Path(subprocess.check_output(
    ["git", "rev-parse", "--show-toplevel"], text=True
).strip())
SRC_DIR = REPO_ROOT / "src"
ENV_FILE = SRC_DIR / "config" / ".env.local"
START_SCRIPT = SRC_DIR / "scripts" / "start_local.sh"

print(f"Repo root:   {REPO_ROOT}")
print(f"Env file:    {ENV_FILE}  (exists={ENV_FILE.exists()})")
print(f"Startup:     {START_SCRIPT}  (exists={START_SCRIPT.exists()})")
print(f"Python:      {sys.executable}")
print()

# -- 1. Load .env.local -------------------------------------------------
loaded = load_dotenv(ENV_FILE, override=True)
print(f"[env]   Loaded {ENV_FILE.name}: {loaded}")

# -- 2. Device = mps ----------------------------------------------------
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    torch.set_default_device("mps")
    os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
    print(f"[torch] default device = mps  (torch {torch.__version__}, fallback=1)")
else:
    DEVICE = torch.device("cpu")
    print(f"[torch] MPS unavailable, falling back to cpu  (torch {torch.__version__})")

_sanity = (torch.randn(4, 4, device=DEVICE) @ torch.randn(4, 4, device=DEVICE)).sum().item()
print(f"[torch] device sanity matmul: {_sanity:.4f}")
print()

# Streaming runner: streams stdout (stderr merged) in real time, returns buffered result.
from collections import namedtuple as _namedtuple

RunResult = _namedtuple("RunResult", ["stdout", "returncode"])

def run_script(args, timeout=None):
    proc = subprocess.Popen(
        [str(START_SCRIPT), *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    buf = []
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
            buf.append(line)
        proc.wait(timeout=timeout)
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait(timeout=10)
        raise
    return RunResult(stdout="".join(buf), returncode=proc.returncode)


# -- 3. Ensure stack is up ---------------------------------------------
print("[stack] checking status...")
status = run_script(["--status"])
if status.returncode != 0:
    print("\n[stack] degraded - running startup (first-run ingestion can take several minutes)...\n")
    startup = run_script([], timeout=1800)
    if startup.returncode != 0:
        raise RuntimeError(f"start_local.sh failed with exit code {startup.returncode}")
else:
    print("[stack] already healthy - skipping startup")
print()

# -- 4. Health check (deep smoke) --------------------------------------
print("[health] running deep smoke test...")
health = run_script(["--status", "--deep"], timeout=300)
if health.returncode != 0:
    raise RuntimeError(f"health check FAILED (exit {health.returncode}) - see output above")
print("[health] PASS - stack is healthy, ready for tests")


## Single test-id eval — Hybrid mode

Runs `ccop-eval evaluate run` on a single test case with retrieval enabled
(default mode = `hybrid`: dense + sparse fusion via Qdrant, then cross-encoder
rerank). Inference goes through the local Ollama-hosted `primus-reasoning`
model.

- **Test ID:** `B03-001` (benchmark B3 — conditional compliance reasoning)
- **Mode:** hybrid (default)
- **Phase:** baseline
- **Scoring:** LLM-as-Judge (B3 is rubric-based)


In [ ]:
# Single test-id eval in Hybrid mode (default)
TEST_ID = "B03-001"

proc = subprocess.Popen(
    ["poetry", "run", "ccop-eval", "evaluate", "run", "--model", "primus-reasoning",
     "--test-ids", TEST_ID],
    cwd=SRC_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)
try:
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait(timeout=600)
except KeyboardInterrupt:
    proc.terminate()
    proc.wait(timeout=10)
    raise
print(f"\nexit_code = {proc.returncode}")
